# OpenAI 适配器

**请确保 OpenAI 库版本为 1.0.0 或更高版本；否则，请参阅旧版文档 [OpenAI 适配器 (旧版)](/docs/integrations/adapters/openai-old/)。**

许多人从 OpenAI 入门，但希望探索其他模型。LangChain 与众多模型提供商的集成使这一过程变得轻松。虽然 LangChain 拥有自己的消息和模型 API，但我们还通过公开一个适配器来适配 LangChain 模型以使用 OpenAI API，从而尽可能轻松地探索其他模型。

目前，这仅处理输出，而不返回其他信息（令牌计数、停止原因等）。

In [1]:
import openai
from langchain_community.adapters import openai as lc_openai

## chat.completions.create

使用此端点创建聊天完成。

**请求**

```json
{
  "model": "gpt-4",
  "messages": [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Who won the world series in 2020?"},
    {"role": "assistant", "content": "The Los Angeles Dodgers won the World Series in 2020."},
    {"role": "user", "content": "Where was it played?"}
  ]
}
```

**参数**

| 参数 | 类型 | 描述 | 必需 | 默认值 |
|---|---|---|---|---|
| `model` | string | 要使用的模型 ID。有关可用模型列表，请参阅[模型文档](https://platform.openai.com/docs/models)。 | 是 |  |
| `messages` | array | 要生成的消息列表。 | 是 |  |
| `frequency_penalty` | number | 取值范围：-2.0 到 2.0。正值会根据它们在文本中出现的频率来降低新词出现的可能性。 | 否 | `0` |
| `logprobs` | boolean | 为每个选择返回词对数概率。 | 否 | `false` |
| `max_tokens` | integer | 此对话中生成令牌的最大数量。 | 否 | `16` |
| `n` | integer | 为每个输入消息生成多少个完成。 | 否 | `1` |
| `presence_penalty` | number | 取值范围：-2.0 到 2.0。正值会根据它们是否已出现在到目前为止的文本中，来鼓励新词的出现。 | 否 | `0` |
| `stop` | string array or string | 指定最多 4 个序列，在遇到这些序列时，模型将停止生成。 | 否 |  |
| `stream` | boolean |  如果设置为 `true`，则在生成过程中将返回部分消息。 | 否 | `false` |
| `temperature` | number | 取值范围：0 到 2。用于控制随机性的值。较高的值（如 0.8）会使输出更随机，而较低的值（如 0.2）会使输出更集中和确定。我们建议将此值设置在 0.7 到 1.0 之间。 | 否 | `1` |
| `top_p` | number | 一个采样概率质量，模型会考虑这些概率质量。因此，1-top_p 的值会排除掉 الأعلى概率的代币。我们建议使用 `top_p` 或 `temperature`，但不要同时使用两者。 | 否 | `1` |

**响应**

```json
{
  "id": "chatcmpl-123",
  "object": "chat.completion",
  "created": 1677652288,
  "model": "gpt-4-0613",
  "choices": [
    {
      "index": 0,
      "message": {
        "role": "assistant",
        "content": "It was played at Globe Life Field in Arlington, Texas."
      },
      "finish_reason": "stop"
    }
  ],
  "usage": {
    "prompt_tokens": 10,
    "completion_tokens": 12,
    "total_tokens": 22
  }
}
```

**字段**

| 字段 | 描述 |
|---|---|
| `id` | 响应的唯一标识符。 |
| `object` | 响应的类型。 |
| `created` | 创建时间戳（以 Unix 时间戳格式表示）。 |
| `model` | 使用的模型。 |
| `choices` | 生成的完成列表。 |
| `usage` | 有关请求中使用的令牌数的信息。 |

**`choices` 字段**

| 字段 | 描述 |
|---|---|
| `index` | 每个完成的索引。 |
| `message` | 关于完成的消息。 |
| `finish_reason` | 生成停止的原因。可能的值为 `stop`（正常停止）、`length`（达到 `max_tokens`）或 `content_filter`（内容被标记为不安全）。 |

**`message` 字段**

| 字段 | 描述 |
|---|---|
| `role` | 发送者的角色。 |
| `content` | 消息的内容。 |

**`usage` 字段**

| 字段 | 描述 |
|---|---|
| `prompt_tokens` | 请求中使用的令牌数。 |
| `completion_tokens` | 生成的令牌数。 |
| `total_tokens` | 请求和响应中使用的总令牌数。 |

In [2]:
messages = [{"role": "user", "content": "hi"}]

原始 OpenAI 调用

In [3]:
result = openai.chat.completions.create(
    messages=messages, model="gpt-3.5-turbo", temperature=0
)
result.choices[0].message.model_dump()

{'content': 'Hello! How can I assist you today?',
 'role': 'assistant',
 'function_call': None,
 'tool_calls': None}

LangChain OpenAI 包装器调用

LangChain 的 OpenAI 包装器允许您使用 OpenAI API。

**安装**

首先，您需要安装 `langchain-openai` 包：

```bash
pip install langchain-openai
```

**基本用法**

以下是如何使用 OpenAI 包装器调用 OpenAI API 的基本示例：

```python
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

# 初始化 ChatOpenAI 模型
# 您可以指定模型名称，例如 "gpt-3.5-turbo" 或 "gpt-4"
# temperature 控制输出的随机性，值越高越随机
# max_tokens 控制生成文本的最大长度
chat = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.7, max_tokens=150)

# 创建一个包含用户消息的列表
messages = [
    HumanMessage(content="请给我讲一个关于太空探险的简短故事。")
]

# 调用模型并获取响应
response = chat.invoke(messages)

# 打印响应内容
print(response.content)
```

**配置**

您可以配置 `ChatOpenAI` 类以满足您的特定需求。

*   **`model`**: 指定要使用的 OpenAI 模型（例如，`"gpt-3.5-turbo"`、`"gpt-4"`）。
*   **`temperature`**: 控制输出的随机性。值越高（例如 0.8），输出越随机；值越低（例如 0.2），输出越确定。
*   **`max_tokens`**: 设置模型生成响应的最大 token 数。
*   **`n`**: 指定要生成的响应数量。
*   **`stop`**: 指定一个停止序列列表。当模型生成其中一个序列时，它将停止生成。
*   **`streaming`**: 如果设置为 `True`，则响应将以流式传输的方式返回，允许您逐步处理输出。

**示例：流式传输**

以下是如何使用流式传输的示例：

```python
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

chat = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.7, streaming=True)

messages = [
    HumanMessage(content="请给我讲一个关于太空探险的简短故事。")
]

# 以流式传输方式调用模型
for chunk in chat.stream(messages):
    print(chunk.content)
```

**使用 LangChain 表达式语言 (LCEL)**

您还可以将 OpenAI 包装器与 LangChain 表达式语言 (LCEL) 结合使用，以创建更复杂的链。

```python
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 定义一个简单的提示模板
prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个乐于助人的助手。"),
    ("user", "{topic} 的简短故事。")
])

# 初始化模型
model = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.7)

# 定义输出解析器
output_parser = StrOutputParser()

# 创建链
chain = prompt | model | output_parser

# 调用链
response = chain.invoke({"topic": "太空探险"})
print(response)
```

**错误处理**

在与 OpenAI API 交互时，可能会遇到各种错误，例如 API 密钥无效、速率限制或模型不可用。建议实现适当的错误处理机制来优雅地处理这些情况。

```python
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from openai import APIError

try:
    chat = ChatOpenAI(model="gpt-3.5-turbo")
    messages = [HumanMessage(content="你好")]
    response = chat.invoke(messages)
    print(response.content)
except APIError as e:
    print(f"OpenAI API 错误: {e}")
except Exception as e:
    print(f"发生意外错误: {e}")
```

**自定义配置**

您可以通过传递其他参数来进一步自定义 `ChatOpenAI` 类的行为，这些参数会直接传递给 OpenAI Python 客户端。有关可用参数的完整列表，请参阅 OpenAI Python 库的文档。

例如，您可以设置 `organization` 或 `client_kwargs`：

```python
from langchain_openai import ChatOpenAI

chat = ChatOpenAI(
    model="gpt-3.5-turbo",
    organization="YOUR_ORG_ID",  # 替换为您的组织 ID
    client_kwargs={
        "api_key": "YOUR_API_KEY",  # 替换为您的 API 密钥
        "base_url": "YOUR_CUSTOM_API_ENDPOINT", # 如果您使用自定义端点
    }
)

print("ChatOpenAI 已使用自定义配置初始化。")

In [4]:
lc_result = lc_openai.chat.completions.create(
    messages=messages, model="gpt-3.5-turbo", temperature=0
)

lc_result.choices[0].message  # Attribute access

{'role': 'assistant', 'content': 'Hello! How can I help you today?'}

In [5]:
lc_result["choices"][0]["message"]  # Also compatible with index access

{'role': 'assistant', 'content': 'Hello! How can I help you today?'}

更换模型提供商

You can swap out model providers to use different LLMs.
您可以更换模型提供商来使用不同的 LLM。

For example, if you want to use `gpt-3.5-turbo` instead of `gpt-4`, you can do so by changing the `model` property in your `config.json` file.
例如，如果您想使用 `gpt-3.5-turbo` 而不是 `gpt-4`，可以通过更改 `config.json` 文件中的 `model` 属性来实现。

```json
{
  "model": "gpt-3.5-turbo"
}
```

You can also swap out providers entirely. For example, if you want to use a local model instead of OpenAI, you can do so by changing the `provider` property in your `config.json` file.
您也可以完全更换提供商。例如，如果您想使用本地模型而不是 OpenAI，可以通过更改 `config.json` 文件中的 `provider` 属性来实现。

```json
{
  "provider": "local"
}
```

You will also need to specify the model you want to use.
您还需要指定您想使用的模型。

```json
{
  "provider": "local",
  "model": "llama2"
}
```

You can find a list of supported providers and models in the documentation.
您可以在文档中找到支持的提供商和模型的列表。

In [6]:
lc_result = lc_openai.chat.completions.create(
    messages=messages, model="claude-2", temperature=0, provider="ChatAnthropic"
)
lc_result.choices[0].message

{'role': 'assistant', 'content': 'Hello! How can I assist you today?'}

## chat.completions.stream

原始 OpenAI 调用

In [7]:
for c in openai.chat.completions.create(
    messages=messages, model="gpt-3.5-turbo", temperature=0, stream=True
):
    print(c.choices[0].delta.model_dump())

{'content': '', 'function_call': None, 'role': 'assistant', 'tool_calls': None}
{'content': 'Hello', 'function_call': None, 'role': None, 'tool_calls': None}
{'content': '!', 'function_call': None, 'role': None, 'tool_calls': None}
{'content': ' How', 'function_call': None, 'role': None, 'tool_calls': None}
{'content': ' can', 'function_call': None, 'role': None, 'tool_calls': None}
{'content': ' I', 'function_call': None, 'role': None, 'tool_calls': None}
{'content': ' assist', 'function_call': None, 'role': None, 'tool_calls': None}
{'content': ' you', 'function_call': None, 'role': None, 'tool_calls': None}
{'content': ' today', 'function_call': None, 'role': None, 'tool_calls': None}
{'content': '?', 'function_call': None, 'role': None, 'tool_calls': None}
{'content': None, 'function_call': None, 'role': None, 'tool_calls': None}


LangChain OpenAI 包装器调用

LangChain 提供了一个 OpenAI 包装器，用于与 OpenAI API 进行交互。此包装器允许您轻松地将 OpenAI 模型集成到您的 LangChain 应用程序中。

以下是如何使用 OpenAI 包装器的示例：

```python
from langchain_openai import OpenAI

# 初始化 OpenAI 包装器
llm = OpenAI(temperature=0.9)

# 调用模型
response = llm.invoke("写一个关于太空探险的短故事。")

print(response)
```

在此示例中：

1.  我们从 `langchain_openai` 导入 `OpenAI` 类。
2.  我们使用 `temperature=0.9` 初始化 `OpenAI` 包装器。`temperature` 参数控制生成文本的随机性。较高的值会产生更多随机的输出，而较低的值会产生更具确定性的输出。
3.  我们使用 `llm.invoke()` 方法调用模型，并提供一个提示作为参数。
4.  响应存储在 `response` 变量中并打印出来。

**配置选项**

OpenAI 包装器支持多种配置选项，包括：

*   `model_name`: 要使用的 OpenAI 模型（例如，`"text-davinci-003"`、`"gpt-3.5-turbo"`）。
*   `temperature`: 控制生成文本的随机性（0 到 2 之间的浮点数）。
*   `max_tokens`: 生成的最大 token 数。
*   `top_p`: 使用核采样（nucleus sampling）时，模型考虑的 token 的累积概率。
*   `frequency_penalty`: 降低模型重复出现相同行的频率的参数（-2.0 到 2.0 之间的浮点数）。
*   `presence_penalty`: 降低模型谈论新主题的频率的参数（-2.0 到 2.0 之间的浮点数）。
*   `openai_api_key`: 您的 OpenAI API 密钥。如果未设置，包装器将尝试从环境变量 `OPENAI_API_KEY` 中读取。

您可以在初始化 `OpenAI` 类时传递这些参数：

```python
from langchain_openai import OpenAI

llm = OpenAI(
    model_name="gpt-3.5-turbo",
    temperature=0.7,
    max_tokens=150,
    openai_api_key="YOUR_API_KEY"
)

response = llm.invoke("解释一下什么是 LangChain。")
print(response)
```

**与聊天模型一起使用**

LangChain 还提供了与 OpenAI 的聊天模型（如 GPT-3.5 Turbo 和 GPT-4）交互的包装器。这些模型通常通过 `ChatOpenAI` 类使用。

```python
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

# 初始化 ChatOpenAI 包装器
chat = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0.8)

# 使用消息列表调用聊天模型
messages = [
    SystemMessage(content="你是一位乐于助人的助手。"),
    HumanMessage(content="给我讲个笑话。")
]

response = chat.invoke(messages)

print(response.content)
```

在此示例中：

1.  我们从 `langchain_openai` 导入 `ChatOpenAI` 类。
2.  我们初始化 `ChatOpenAI` 包装器，指定模型名称和温度。
3.  我们创建一个消息列表，其中包含一个 `SystemMessage`（用于设置助手的行为）和一个 `HumanMessage`（用户的输入）。
4.  我们使用 `chat.invoke()` 方法调用聊天模型，并传递消息列表。
5.  响应的内容存储在 `response.content` 中并打印出来。

OpenAI 包装器是利用 OpenAI 强大功能构建 LangChain 应用程序的关键组件。

In [8]:
for c in lc_openai.chat.completions.create(
    messages=messages, model="gpt-3.5-turbo", temperature=0, stream=True
):
    print(c.choices[0].delta)

{'role': 'assistant', 'content': ''}
{'content': 'Hello'}
{'content': '!'}
{'content': ' How'}
{'content': ' can'}
{'content': ' I'}
{'content': ' assist'}
{'content': ' you'}
{'content': ' today'}
{'content': '?'}
{}


更换模型提供商

You can swap out the model provider for any component that uses a model.
您可以更换使用模型的任何组件的模型提供商。

For example, if you want to use OpenAI's models instead of the default ones, you can do so by creating a new `OpenAI` instance and passing it to the component.
例如，如果您想使用 OpenAI 的模型而不是默认模型，可以通过创建一个新的 `OpenAI` 实例并将其传递给组件来实现。

```jsx
import { ChatCompletion } from "@ai-chatbot/core/completion";
import OpenAI from "@ai-chatbot/openai";

const openai = new OpenAI({
  apiKey: process.env.OPENAI_API_KEY,
});

const chatCompletion = new ChatCompletion({
  model: openai.chat.completions,
});
```

You can also swap out the model provider for a specific component, such as the `ChatCompletion` component.
您还可以为特定组件（如 `ChatCompletion` 组件）更换模型提供商。

```jsx
import { ChatCompletion } from "@ai-chatbot/core/completion";
import OpenAI from "@ai-chatbot/openai";

const openai = new OpenAI({
  apiKey: process.env.OPENAI_API_KEY,
});

const chatCompletion = new ChatCompletion({
  model: openai.chat.completions,
});
```

This will use OpenAI's models for chat completions.
这将使用 OpenAI 的模型进行聊天补全。

If you want to use a different model provider, you can do so by creating a new instance of that provider and passing it to the component.
如果您想使用其他模型提供商，可以通过创建该提供商的新实例并将其传递给组件来实现。

For example, if you want to use Anthropic's models, you can do so by creating a new `Anthropic` instance and passing it to the component.
例如，如果您想使用 Anthropic 的模型，可以通过创建一个新的 `Anthropic` 实例并将其传递给组件来实现。

```jsx
import { ChatCompletion } from "@ai-chatbot/core/completion";
import Anthropic from "@ai-chatbot/anthropic";

const anthropic = new Anthropic({
  apiKey: process.env.ANTHROPIC_API_KEY,
});

const chatCompletion = new ChatCompletion({
  model: anthropic.completions,
});
```

This will use Anthropic's models for chat completions.
这将使用 Anthropic 的模型进行聊天补全。

In [9]:
for c in lc_openai.chat.completions.create(
    messages=messages,
    model="claude-2",
    temperature=0,
    stream=True,
    provider="ChatAnthropic",
):
    print(c["choices"][0]["delta"])

{'role': 'assistant', 'content': ''}
{'content': 'Hello'}
{'content': '!'}
{'content': ' How'}
{'content': ' can'}
{'content': ' I'}
{'content': ' assist'}
{'content': ' you'}
{'content': ' today'}
{'content': '?'}
{}
